## Unlabeled Data

This Notebook uses the MassSpecGym datasets to create test data for the unlabeled input. This data includes:
1. Unlabeled candidates (for precomputation)
2. Spectra (for online retrieval)

both will be constructed with [MassSpecGym_retrieval_candidates_formula.json](https://huggingface.co/datasets/roman-bushuiev/MassSpecGym/blob/main/data/molecules/MassSpecGym_retrieval_candidates_formula.json) and the official [MassSpecGym dataset](https://huggingface.co/datasets/roman-bushuiev/MassSpecGym)

In [27]:
import json
import pandas as pd
from jestr.data.datasets import MassSpecDataset, ExpandedRetrievalDataset


#### 1. Load dataset
- uses the original workflow to extend MassSpecGym dataset with candidates
- with that build than the sepparate datasets

In [28]:
# Load ExpandedRetrievalDataset
# pass spectra_view default 
spectra_view = "SpecBinnerLog"
erd = ExpandedRetrievalDataset(spectra_view=spectra_view, pth = None, candidates_pth= None)
print(f"ExpandedRetrievalDataset instantiated. spectra metadata rows: {len(erd.instance.metadata)}")


No path provided for mass spectrum dataset. Downloading MassSpecGym dataset from HuggingFace Hub...
Parent calss metadata columns: ['identifier', 'smiles', 'inchikey', 'formula', 'precursor_formula', 'parent_mass', 'precursor_mz', 'adduct', 'instrument_type', 'collision_energy', 'fold', 'simulation_challenge']
No path provided downloads from hugging face and stores file in .cache/huggingface/datasets/molecules/MassSpecGym_retrieval_candidates_formula.json
Skipping MassSpecGymID0078299; empty candidate set
Skipping MassSpecGymID0078599; empty candidate set
Skipping MassSpecGymID0189021; empty candidate set
Skipping MassSpecGymID0189022; empty candidate set
Skipping MassSpecGymID0189031; empty candidate set
Skipping MassSpecGymID0195390; empty candidate set
Skipping MassSpecGymID0202855; empty candidate set
Skipping MassSpecGymID0212102; empty candidate set
Skipping MassSpecGymID0225264; empty candidate set
Skipping MassSpecGymID0227974; empty candidate set
Skipping MassSpecGymID0230938;

In [15]:
print(f"length of spectra: {len(erd.spectra)}")
print(f"length of dataset: {len(erd)}")
print("first spectrum metadata:", erd.spectra[0])
print(" metadata columns:", erd.metadata.columns.tolist())
print("Spectrum candidates (spec_cand):", len(erd.spec_cand))
print("First 5 entries of spec_cand:", erd.spec_cand[:5])
counter = 0
current = 0
spec_indexes_with_candidates = list()
for i, candtuple in enumerate(erd.spec_cand):
    metaindex = candtuple[0]
    if current != metaindex:
        counter += 1
        current = metaindex
        spec_indexes_with_candidates.append(metaindex)
print(f"Number of spectra with candidates: {counter}")

length of spectra: 231104
length of dataset: 571118
first spectrum metadata: <matchms.Spectrum.Spectrum object at 0x7fde76f30d00>
 metadata columns: ['identifier', 'smiles', 'inchikey', 'formula', 'precursor_formula', 'parent_mass', 'precursor_mz', 'adduct', 'instrument_type', 'collision_energy', 'fold', 'simulation_challenge']
Spectrum candidates (spec_cand): 571118
First 5 entries of spec_cand: [(168, 'CC(C)[C@@H]1C(=O)N([C@H](C(=O)O[C@@H](C(=O)N([C@H](C(=O)O[C@@H](C(=O)N([C@H](C(=O)O1)CC2=CC=CC=C2)C)C(C)C)CC3=CC=CC=C3)C)C(C)C)CC4=CC=CC=C4)C', True), (168, 'CC(C)C1C(=O)OC(Cc2ccccc2)C(=O)N(C)C(C(C)C)C(=O)OC(Cc2ccccc2)C(=O)N(C)C(C(C)C)C(=O)OC(Cc2ccccc2)C(=O)N1C', False), (168, 'COC(=O)[C@@H]1CCCN1C(=O)[C@H](C)NC(=O)[C@H](CC(C)C)NC(=O)/C=C/[C@H]1O[C@@H](C)[C@@H](OCc2ccccc2)[C@@H](OCc2ccccc2)[C@@H]1OCc1ccccc1', False), (168, 'C=CCO[C@@]12Oc3ccc(OC(=O)NCC)cc3[C@H]3[C@H](CCCCO)[C@@H](CCCCO)C=C(C(=NOCC)C[C@@H]1N(Cc1cccc4ccccc14)C(=O)OC)[C@H]32', False), (168, 'COCCCN1C(=O)COc2ccc(N(C(=O)[C@

#### 3. Split dataset

In [6]:
# Creates two datasets from the ExpandedRetrievalDataset instance,
#  one for unlabeled candidates and one for spectra information
# only take data entries with test labels

# 1. Unlabeled candidates (for precomputation)
# Select all candidates in the found Dataset by keeping their order
# export a list of list of candidates to a json file
# collect candidates into a list of lists (one list of candidates per spectrum)
from collections import defaultdict
import json

def build_list_of_lists_of_candidates(erd):
    grouped = defaultdict(list)

    # spec_cand contains tuples: (spec_idx, cand_smiles, label)
    for i, (spec_idx, cand_smiles, label) in enumerate(erd.spec_cand):
        grouped[spec_idx].append(cand_smiles)
    # convert to list of lists is order stable
    list_of_lists = list(grouped.values())
    return list_of_lists

list_of_lists_of_candidates = build_list_of_lists_of_candidates(erd)

with open("data/sample/list_of_lists_of_candidates2.json", "w") as f:
    json.dump(list_of_lists_of_candidates, f, indent=2)

print("Saved", len(list_of_lists_of_candidates), "candidate lists.")


Saved 3142 candidate lists.


In [25]:
# 2. Spectra (for online retrieval)
# Select all spectra and their information in the found Dataset
# Exclude candidates/target info and keep only a small set of metadata fields
from matchms import Spectrum
from matchms.exporting import save_as_json
import numpy as np
import pandas as pd

def _normalize_value(v):
    # Convert numpy/pandas types to native Python types so json can serialize them
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, (np.generic,)):
        try:
            return v.item()
        except Exception:
            return v
    if isinstance(v, pd.Series):
        return v.to_dict()
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return v

spectra_out = []

for spec_idx in spec_indexes_with_candidates:
    spec = erd.instance.spectra[spec_idx]
    metadata = erd.instance.metadata.iloc[spec_idx]

    raw_meta = {
        "identifier": metadata.get("identifier", ""),
        "precursor_formula": metadata.get("precursor_formula", ""),
        "parent_mass": metadata.get("parent_mass", None),
        "precursor_mz": spec.get("precursor_mz", None),
        "peaks_mz": spec.get("peaks_mz", []),
        "adduct": metadata.get("adduct", ""),
        "instrument_type": metadata.get("instrument_type", ""),
        "collision_energy": metadata.get("collision_energy", None),
        "simulation_challenge": metadata.get("simulation_challenge", None),
    }
    # Normalize any numpy/pandas scalars or arrays to plain Python types
    clean_meta = {k: _normalize_value(v) for k, v in raw_meta.items()}

    # matchms Spectrum expects numpy arrays for peaks; metadata should be JSON-serializable
    new_spec = Spectrum(
        mz=np.array(spec.peaks.mz, dtype=float),
        intensities=np.array(spec.peaks.intensities, dtype=float),
        metadata=clean_meta
    )

    spectra_out.append(new_spec)

# Save as matchms-compatible JSON
output_path = "data/sample/expanded_retrieval_dataset.json"
save_as_json(spectra_out, output_path)

print(f"Saved {len(spectra_out)} spectra to {output_path}")

/tmp/ipykernel_2136384/556470942.py:21: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(v):


Saved 3142 spectra to data/sample/expanded_retrieval_dataset.json


In [ ]:
print(spec_indexes_with_candidates[20])

9974


### 4. Create a Test split
- for fast testing

In [26]:
#Create unlabeled candidates test set
list_of_lists_of_candidates_test = list_of_lists_of_candidates[:3]  # Take first 3 spectra's candidates for testing
#Create spectrum test set
list_of_spectra_test = spectra_out[:3]  # Take first 3 spectra for testing
#Save test sets
with open("data/sample/list_of_lists_of_candidates_test.json", "w") as f:
    json.dump(list_of_lists_of_candidates_test, f, indent=2)    
save_as_json(list_of_spectra_test, "data/sample/expanded_retrieval_dataset_test2.json")
print("Saved test sets.")


Saved test sets.


### 5. Create Labeled Dataset

In [40]:
# Spectra to smiles test set
#make tsv of spectra metadata with test labels
# add smiles inchikey formula from metadata

#import 
rows = []
for i, spec in enumerate(list_of_spectra_test):
    meta = spec.metadata
    identifier = meta.get("identifier", "")
    precursor_formula = meta.get("precursor_formula", "")
    parent_mass = meta.get("parent_mass", None)
    precursor_mz = meta.get("precursor_mz", None)
    adduct = meta.get("adduct", "")
    instrument_type = meta.get("instrument_type", "")
    collision_energy = meta.get("collision_energy", None)
    simulation_challenge = meta.get("simulation_challenge", None)
    # from original list
    smiles = erd.metadata.iloc[spec_indexes_with_candidates[i]].get("smiles", "")
    inchikey = erd.metadata.iloc[spec_indexes_with_candidates[i]].get("inchikey", "")
    formula = erd.metadata.iloc[spec_indexes_with_candidates[i]].get("formula", "")
    fold = erd.metadata.iloc[spec_indexes_with_candidates[i]].get("fold", None)
    #order Parent class metadata columns: ['identifier', 'mzs', 'intensities', smiles', 'inchikey', 'formula', 'precursor_formula', 'parent_mass', 'precursor_mz', 'adduct', 'instrument_type', 'collision_energy', 'fold', 'simulation_challenge']
    rows.append({
        "identifier": identifier,
        "mzs": ",".join(str(x) for x in spec.peaks.mz.tolist()),
        "intensities": ",".join(str(x) for x in spec.peaks.intensities.tolist()),
        "smiles": smiles,
        "inchikey": inchikey,
        "formula": formula,
        "precursor_formula": precursor_formula,
        "parent_mass": parent_mass,
        "precursor_mz": precursor_mz,
        "adduct": adduct,
        "instrument_type": instrument_type,
        "collision_energy": collision_energy,
        "fold": fold,
        "simulation_challenge": simulation_challenge
    })
df = pd.DataFrame(rows)
df.to_csv("data/sample/spectra_metadata_test.tsv", sep="\t", index=False)
print("Saved spectra metadata test set to TSV.")


Saved spectra metadata test set to TSV.


In [ ]:
# create candidates_retieval test set not needed since list of candidates can be big